## **Fase 00 - Explicação Conceitual Teórioca**

Explicação do processo de algoritmo evolucionário aplicado as mecânicas do jogo **Flappy Bird** em detalhes.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd

# Estilo dos gráficos
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(42)
print("Setup concluído.")

Setup concluído.


# Matemática da Computação Evolucionária aplicada ao Flappy Bird

> **Documento didático — Fase Zero**
> Tema 3: Algoritmos Genéticos (GA) aplicados ao controle autônomo de um agente no jogo Flappy Bird

---

## Sumário

- **PARTE A — FUNDAMENTOS**
  - A.1 O que é um Algoritmo Genético
  - A.2 Analogia com evolução biológica
  - A.3 O que é um vetor
  - A.4 Número aleatório uniforme $U(0,1)$
  - A.5 Distribuição normal $\mathcal{N}(0,\sigma^2)$
  - A.6 Somatório $\sum$
  - A.7 Função de ativação

- **PARTE B — REPRESENTAÇÃO DO PÁSSARO**
  - B.1 Genoma (vetor de 6 genes)
  - B.2 Entradas sensoriais + normalização
  - B.3 Soma ponderada $z$
  - B.4 Sigmoide
  - B.5 Regra de decisão
  - B.6 Exemplo completo

- **PARTE C — FÍSICA DO JOGO**
- **PARTE D — FITNESS**
- **PARTE E — MORTE E SELEÇÃO**
- **PARTE F — CRUZAMENTO**
- **PARTE G — MUTAÇÃO**
- **PARTE H — NOVA GERAÇÃO**
- **PARTE I — CICLO COMPLETO**
- **PARTE J — GLOSSÁRIO**

# PARTE A — FUNDAMENTOS

Nesta parte não falamos de Flappy Bird ainda. Aqui construímos o alicerce: os conceitos e a notação que serão usados no resto do documento. Se você pular esta parte, o resto não vai fazer sentido.

---

## A.1 O que é um Algoritmo Genético, em linguagem de leigo

**Em termos simples:** um Algoritmo Genético (GA) é uma forma de resolver problemas por tentativa e erro organizada, imitando como a natureza faz para melhorar espécies ao longo de gerações.

Imagine que você precisa descobrir uma combinação de números (uma "receita") que faça um robô jogar bem um jogo. Você não sabe quais números usar. Poderia testar todas as combinações possíveis, mas o número de combinações é astronômico — levaria séculos. Em vez disso, você faz o seguinte:

1. Cria 50 receitas aleatórias.
2. Testa todas no jogo e dá uma nota para cada uma (nota alta = jogou bem).
3. As receitas com nota alta têm mais chance de "se reproduzir": você pega pedaços de duas boas receitas e monta uma receita nova.
4. Adiciona pequenos erros aleatórios nas receitas novas (para não ficarem todas iguais).
5. Descarta as receitas ruins e repete com as novas.

Depois de dezenas de repetições, as receitas ficam cada vez melhores — não porque você programou a solução, mas porque o processo de seleção + reprodução + erro foi empurrando a população na direção certa.

**Por que isso funciona?** Porque em quase todo problema existe uma "paisagem" onde soluções parecidas tendem a ter desempenho parecido. Se você pega duas soluções boas e mistura, frequentemente a mistura também é boa. E pequenas perturbações ocasionalmente descobrem um "degrau" ainda mais alto.

**Efeito prático:**
- Se você **aumentar** o número de indivíduos por geração, o algoritmo fica mais lento mas mais robusto.
- Se você **diminuir**, fica mais rápido mas pode não encontrar boas soluções.

**Resumo da seção:** Algoritmo Genético é resolver problemas por tentativa organizada, imitando evolução: cria população aleatória → avalia → seleciona melhores → reproduz com mistura → muta → repete. Não programamos a solução; deixamos ela emergir.

## A.2 Analogia com evolução biológica

**Em termos simples:** cada termo técnico do GA tem um correspondente na biologia. Entender essa correspondência é a chave para não se perder.

| Termo no GA | Correspondente biológico | O que significa no nosso Flappy Bird |
|-------------|--------------------------|--------------------------------------|
| **Gene** | Um trecho de DNA | Um número real (ex.: `0.31`) |
| **Genoma** (ou indivíduo) | O DNA completo de um ser vivo | Um vetor de 6 números |
| **População** | O conjunto de seres vivos de uma espécie | 50 pássaros simultâneos |
| **Geração** | Uma "rodada" de vida | Um ciclo completo: avaliar → reproduzir |
| **Fitness** (aptidão) | Quão bem o ser vivo sobrevive | Pontuação obtida no jogo |
| **Seleção natural** | Quem sobrevive para reproduzir | Escolhemos os de maior fitness como pais |
| **Cruzamento** (crossover) | Reprodução sexuada | Misturamos os genes de dois pais |
| **Mutação** | Erro na cópia do DNA | Perturbamos aleatoriamente um gene |
| **Elitismo** | Os mais fortes sempre sobrevivem | Copiamos os melhores intactos para a próxima geração |

**Como isso se lê na prática:** a natureza não "sabe" qual é o melhor DNA. Ela apenas faz com que os indivíduos mais bem adaptados tenham mais filhos. Os filhos herdam metade do DNA de cada pai, e erros de cópia ocasionais geram novidade. Ao longo de milhões de anos, isso produz olhos, asas, cérebros. No nosso caso, "milhões de anos" viram "dezenas de gerações de computador" — o princípio é o mesmo.

**Efeito prático:**
- **Sem cruzamento**, cada indivíduo só pode melhorar por mutação: evolução lenta.
- **Sem mutação**, todos os filhos ficam iguais aos pais: a evolução estagnaria em poucas gerações.
- **Sem seleção**, todos se reproduzem igualmente: não há pressão para melhorar.

**Resumo da seção:** Todo termo do GA tem gêmeo biológico. Gene é número, genoma é vetor, população é conjunto de pássaros, fitness é nota, seleção é escolher pais, cruzamento é reprodução, mutação é erro de cópia, elitismo é garantir que o melhor nunca morra. Os três pilares que fazem a evolução andar são: seleção, cruzamento e mutação — sem qualquer um deles, o algoritmo não funciona bem.